# 02_pipeline — config-driven orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and beginner friendly. The default demo reads two real source DataFrames, configures source guardrails, transforms those sources into two target DataFrames, declares target table configs only after the target DataFrames exist, runs target guardrails, writes targets, and records lineage plus run-summary evidence.

To adapt the template, edit the clearly marked **USER EDIT SECTION** blocks. Add source DataFrame reads before `SOURCE_TABLES`, add transformations before `TARGET_TABLES`, and then add target dictionaries for DataFrames that already exist. Do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, write, lineage, or runtime-summary orchestration code.

FabricOps enriches source and target entries with guardrail defaults and write defaults before running profiling, schema validation, profile behavior enforcement, DQ enforcement, catalogue evidence, writes, lineage, and runtime summary from the config lists.


## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.


In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables for agreement selection, table-config preparation, guardrail orchestration, explicit target writes, lineage, and runtime-summary evidence.


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit.config import _current_audit_timestamp

from fabricops_kit import (
    get_selected_agreement,
    prepare_pipeline_table_configs,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    run_table_guardrails,
    widget_select_agreement,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select data agreement and capture run context

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`. The run context values are reused by guardrail evidence, lineage, and runtime summary writes.


In [ ]:
PIPELINE_STARTED_AT = _current_audit_timestamp(config=CONFIG)
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = RUN_CONTEXT.runtime_metadata.get("currentNotebookName", "02_pipeline")

widget_select_agreement(
    CONFIG,
    env_name=ENV_NAME,
    spark_session=spark,
    metadata_schema=METADATA_SCHEMA,
    register_notebook=True,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. USER EDIT SECTION — read source DataFrames

Read each source DataFrame first, using the existing FabricOps IO helper that matches where the data lives. The default demo reads two smoke source tables created by `example_pipeline_smoke_test.ipynb` so users can see a many-source pipeline before configuring guardrails or targets.

Optional CSV, parquet, Excel, warehouse, and Spark table examples are shown as commented alternatives. Keep the primary path concrete: read DataFrames here, transform them later, and only declare target tables after target DataFrames exist.


In [ ]:
df_orders = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    "source",
    "smoke_src_orders_happy",
    schema=SOURCE_SCHEMA,
    spark_session=spark,
)

df_customers = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    "source",
    "smoke_src_customers_happy",
    schema=SOURCE_SCHEMA,
    spark_session=spark,
)

# CSV file in a configured Lakehouse target:
# df_orders = read_lakehouse_csv(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/orders.csv",
#     spark_session=spark,
#     header=True,
# )

# Parquet file or folder in a configured Lakehouse target:
# df_orders = read_lakehouse_parquet(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/orders.parquet",
#     verbose=True,
#     spark_session=spark,
# )

# Excel file in a configured Lakehouse target:
# df_customers = read_lakehouse_excel(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/customers.xlsx",
#     sheet_name=0,
#     spark_session=spark,
# )

# Warehouse table:
# df_orders = read_warehouse_table(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "dbo",
#     "orders",
#     spark_session=spark,
# )

# Custom Spark table reference:
# df_customers = spark.read.table("database.customers")


## 5. USER EDIT SECTION — configure source guardrails

Configure guardrails and catalogue evidence for DataFrames that were already read. FabricOps derives governance `dataset_name` from `table_name` and uses `layer` as the default governance `stage`, so normal source examples do not need `dataset_name`.

Valid FabricOps layer/stage concepts:

- `source` = raw/source lakehouse table.
- `unified` = cleaned/conformed lakehouse table.
- `product` = curated product or warehouse output.
- `metadata` = governance evidence lakehouse. It is normally configured in `00_env_config` and is not usually selected as a business source table.

To add sources, read another DataFrame above and add another dictionary to `SOURCE_TABLES` with a unique `key`. Do not copy profiling, schema, freshness, profile behavior, DQ, or catalogue-evidence code.

Advanced override support: add `dataset_name`, `stage`, or `dq_preset` inside a specific source table config only when that table needs to differ from the guardrail defaults.


In [ ]:
SOURCE_TABLES = [
    {
        "key": "orders",
        "df": df_orders,
        "layer": "source",
        "table_name": "smoke_src_orders_happy",
        "watermark_column": "order_date",
        "expected_schema": {
            "order_id": "bigint",
            "customer_id": "bigint",
            "order_date": "date",
            "ingestion_ts": "timestamp",
            "status": "string",
            "order_amount": "double",
            "country_code": "string",
        },
    },
    {
        "key": "customers",
        "df": df_customers,
        "layer": "source",
        "table_name": "smoke_src_customers_happy",
        "watermark_column": "effective_date",
        "expected_schema": {
            "customer_id": "bigint",
            "customer_name": "string",
            "customer_segment": "string",
            "customer_country_code": "string",
            "effective_date": "date",
            "ingestion_ts": "timestamp",
        },
        "dq_preset": "skip",
    },
]

# Optional advanced per-table guardrail overrides, only when needed:
# "dataset_name": "governance_dataset_override",
# "stage": "source",
# "dq_preset": "approved_rules",


## 6. Source guardrail defaults

These are the default guardrails applied to every source table. Most users should leave them as shown. Override a value inside a `SOURCE_TABLES` entry only when one source table needs a different schema rule, freshness rule, load behavior, DQ rule, profile distribution, or excluded column.


In [ ]:
DEFAULT_SOURCE_GUARDRAILS = {
    # Schema preset options:
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "strict" = require the schema to match exactly
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "allow_new_columns",

    # Load behavior guardrail options:
    #   "append" = protect existing history
    #   "overwrite" = accept full refresh/rebuild as the new state
    #   "skip" = skip only profile behavior enforcement
    "load_behavior": "append",

    # Freshness guardrail options:
    #   freshness_column = date/timestamp column that proves latest data arrived
    #   freshness_max_lag_days = allowed lag from today's date
    #   freshness_severity = "blocking" or "warning"
    "freshness_column": "order_date",
    "freshness_max_lag_days": 1,
    "freshness_severity": "blocking",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile guardrail settings.
    "distribution_columns": ["status", "order_amount", "country_code"],
    "exclude_columns": None,
}


## 7. Prepare source table configs

FabricOps derives beginner-friendly governance defaults and adds the default source guardrails to each pre-loaded source DataFrame. Most users do not need to edit this section.


In [ ]:
SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    DEFAULT_SOURCE_GUARDRAILS,
    table_role="source",
)

# Use prepared DataFrame aliases in transformations so defaults and per-table overrides are centralized.
df_orders = SOURCE_CONFIG_BY_KEY["orders"]["df"]
df_customers = SOURCE_CONFIG_BY_KEY["customers"]["df"]


## 8. Optional: inspect a source schema

Run this cell while authoring if you want Spark to show the actual source schema before you finish `expected_schema` in the USER EDIT SECTION.


In [ ]:
# Optional authoring check: inspect the Spark schema before writing expected_schema.
df_source_01.printSchema()


## 9. Run source guardrails before transformation

FabricOps runs profiling, schema validation, profile behavior checks, DQ checks, catalogue evidence, and optional guardrail stopping through `run_table_guardrails`. Before catalogue evidence is written to `METADATA_DATA_CATALOGUE`, FabricOps schema-aligns generated evidence columns such as row counts, DQ counts, percentages, timestamps, and booleans to the metadata table schema. Source guardrails run before transformation, and most users should not need to customize this orchestration code.


In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(source_guardrail_results["summary"])

# Runtime summary and lineage cells reuse these package-generated evidence objects.
source_schema_results = source_guardrail_results["schema_results"]
source_freshness_results = source_guardrail_results["freshness_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


## 10. USER EDIT SECTION — DIY transformations

This is the only section where most users write business transformation logic. Create one target DataFrame for each target table you plan to publish. FabricOps guardrails and audit columns are handled in later sections.


The demo keeps geography explicit: `orders.country_code` is transaction/order geography, while `customers.customer_country_code` is customer-profile geography. The join selects both columns by distinct names so the enriched target has no duplicate `country_code` columns. Summaries group by the order-level `country_code`.


In [ ]:
df_orders_enriched = (
    df_orders.alias("orders")
    .join(df_customers.alias("customers"), on="customer_id", how="left")
    .select(
        F.col("orders.order_id"),
        F.col("orders.customer_id"),
        F.col("customers.customer_name"),
        F.col("customers.customer_segment"),
        F.col("orders.country_code"),
        F.col("customers.customer_country_code"),
        F.col("orders.order_date"),
        F.col("orders.ingestion_ts"),
        F.col("orders.status"),
        F.col("orders.order_amount"),
    )
    .withColumn(
        "order_amount_band",
        F.when(F.col("order_amount") >= F.lit(100), F.lit("high"))
        .when(F.col("order_amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)

df_orders_summary = (
    df_orders_enriched
    .groupBy("customer_segment", "country_code")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("order_amount").alias("total_order_amount"),
        F.max("order_date").alias("latest_order_date"),
    )
)


## 11. USER EDIT SECTION — configure target tables after transformation

Declare target table configs only after the transformed target DataFrames exist. Update each target `key`, `df`, `layer`, `table_name`, `write_mode`, `watermark_column`, and `expected_schema`. FabricOps derives governance `dataset_name` from `table_name`, uses `layer` as the default governance `stage` and write layer, uses `table_name` as the default target write name, and uses `lakehouse` as the default target kind.

To add targets, create another DataFrame in the transform section, then add another dictionary to `TARGET_TABLES` with a unique `key`. Do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, or write orchestration code.

Advanced override support: add `dataset_name`, `stage`, `target_layer`, `target_name`, `target_kind`, `dq_preset`, or `kind` inside a specific target table config only when that table needs to differ from the guardrail or write defaults.


In [ ]:
TARGET_TABLES = [
    {
        "key": "orders_enriched",
        "df": df_orders_enriched,
        "layer": "unified",
        "table_name": "smoke_unified_orders_enriched",
        "write_mode": "overwrite",
        "watermark_column": "order_date",
        "expected_schema": {
            "order_id": "bigint",
            "customer_id": "bigint",
            "customer_name": "string",
            "customer_segment": "string",
            "country_code": "string",
            "customer_country_code": "string",
            "order_date": "date",
            "ingestion_ts": "timestamp",
            "status": "string",
            "order_amount": "double",
            "order_amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    },
    {
        "key": "orders_summary",
        "df": df_orders_summary,
        "layer": "product",
        "table_name": "smoke_product_orders_summary",
        "write_mode": "overwrite",
        "watermark_column": "latest_order_date",
        "expected_schema": {
            "customer_segment": "string",
            "country_code": "string",
            "order_count": "bigint",
            "total_order_amount": "double",
            "latest_order_date": "date",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
        "freshness_column": "latest_order_date",
        "distribution_columns": ["customer_segment", "country_code"],
        "dq_preset": "skip",
    },
]

# Optional advanced per-table overrides, only when needed:
# "dataset_name": "governance_dataset_override",
# "stage": "product",
# "target_layer": "product",
# "target_name": "written_table_name_override",
# "target_kind": "warehouse",
# "dq_preset": "approved_rules",
# "kind": "warehouse",
# "partition_by": ["order_date"],
# "repartition_by": ["customer_id"],
# "options": {"overwriteSchema": "true"},


## 12. Target guardrail and write defaults

These are the default guardrails and write options applied to every target table. Most users should leave them as shown. Override a value inside a `TARGET_TABLES` entry only when one target table needs a different schema rule, freshness rule, load behavior, DQ rule, profile distribution, excluded column, or write option.


In [ ]:
TARGET_LAYER_SCHEMAS = {
    "source": SOURCE_SCHEMA,
    "unified": UNIFIED_SCHEMA,
    "product": PRODUCT_SCHEMA,
    "metadata": METADATA_SCHEMA,
}

DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS = {
    # Schema preset options:
    #   "strict" = require the schema to match exactly
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "strict",

    # Load behavior guardrail options:
    #   "append" = protect existing history
    #   "overwrite" = accept full refresh/rebuild as the new state
    #   "skip" = skip only profile behavior enforcement
    "load_behavior": "overwrite",

    # Freshness guardrail options:
    #   freshness_column = date/timestamp column that proves latest data arrived
    #   freshness_max_lag_days = allowed lag from today's date
    #   freshness_severity = "blocking" or "warning"
    "freshness_column": "order_date",
    "freshness_max_lag_days": 1,
    "freshness_severity": "blocking",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile guardrail settings.
    "distribution_columns": ["status", "order_amount", "country_code"],
    "exclude_columns": None,

    # Write mode options for Lakehouse targets:
    #   "overwrite" = replace the target table
    #   "append" = add rows to the target table
    #   "errorifexists" = fail if the target table already exists
    #   "ignore" = skip the write if the target table already exists
    # Warehouse writes use Spark connector modes such as "overwrite" or "append".
    "write_mode": "overwrite",

    # Optional Lakehouse write options.
    "partition_by": None,
    "repartition_by": None,
    "options": {"overwriteSchema": "true"},

    # Target kind options:
    #   "lakehouse" = write a Lakehouse Delta table
    #   "warehouse" = write a Fabric Warehouse table
    "kind": "lakehouse",
}


## 13. Prepare target table configs

Do not edit this section for normal target tables. FabricOps adds runtime audit columns, applies `DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS`, derives target write metadata, and keeps downstream guardrails and writes driven by `TARGET_TABLES`.

The `PRIMARY_TARGET_CONFIG` variable created here is a convenience alias for the first target dataset name used when writing pipeline lineage.


In [ ]:
TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS,
    table_role="target",
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
)

# Convenience alias keeps the default lineage dataset name easy to read.
PRIMARY_TARGET_CONFIG = TARGET_CONFIG_BY_KEY["orders_enriched"]


## 14. Run target guardrails before writes

Target profiling, schema validation, profile behavior checks, DQ checks, and catalogue evidence run for every config in `TARGET_TABLES`. Target writes do not happen unless `run_table_guardrails(..., stop_on_failure=True)` completes.


In [ ]:
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(target_guardrail_results["summary"])

target_schema_results = target_guardrail_results["schema_results"]
target_freshness_results = target_guardrail_results["freshness_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]


## 15. Write target tables

Only after all configured target guardrails pass, explicitly write each target DataFrame to its configured Fabric target. Keep this section visible because it is the point where data is published.


In [ ]:
target_write_status = {}
for target_config in TARGET_TABLES:
    target_key = target_config["key"]
    target_kind = str(target_config.get("target_kind", target_config.get("kind", "lakehouse"))).lower()
    target_layer = target_config.get("target_layer", target_config.get("layer", "unified"))
    target_name = target_config.get("target_name", target_config.get("table_name", target_key))
    target_mode = target_config.get("write_mode", target_config.get("mode", "overwrite"))

    if target_kind == "lakehouse":
        write_lakehouse_table(
            target_config["df"],
            CONFIG,
            ENV_NAME,
            target_layer,
            target_name,
            schema=target_config.get("schema", TARGET_LAYER_SCHEMAS.get(target_layer)),
            mode=target_mode,
            partition_by=target_config.get("partition_by"),
            repartition_by=target_config.get("repartition_by"),
            options=target_config.get("options", {"overwriteSchema": "true"} if target_mode == "overwrite" else None),
        )
    elif target_kind == "warehouse":
        write_warehouse_table(
            target_config["df"],
            CONFIG,
            ENV_NAME,
            target_layer,
            target_config.get("schema", "dbo"),
            target_name,
            mode=target_mode,
        )
    else:
        raise ValueError(f"Unsupported target kind for {target_key}: {target_kind}")

    target_write_status[target_key] = "written"


## 16. USER EDIT SECTION — lineage relationships

Describe how configured source tables produce configured target tables. Each relationship uses source and target keys from `SOURCE_TABLES` and `TARGET_TABLES`.

A relationship can contain one or more source keys and one or more target keys. FabricOps expands many-to-many relationships into table-level lineage rows for every source-target pair.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["orders", "customers"],
        "targets": ["orders_enriched", "orders_summary"],
        "operation": "join orders to customers, enrich orders, and summarize by customer attributes",
        "description": (
            f"{SOURCE_CONFIG_BY_KEY['orders']['table_name']} and "
            f"{SOURCE_CONFIG_BY_KEY['customers']['table_name']} produce "
            f"{TARGET_CONFIG_BY_KEY['orders_enriched']['table_name']} and "
            f"{TARGET_CONFIG_BY_KEY['orders_summary']['table_name']}."
        ),
    },
]


## 17. Write lineage

FabricOps writes lineage evidence after target writes complete.


In [ ]:
lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=PRIMARY_TARGET_CONFIG["dataset_name"],
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 18. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=_current_audit_timestamp(config=CONFIG),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_freshness_results=source_freshness_results,
    target_freshness_results=target_freshness_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
